In [1]:
# Install packages
!pip install langchain_core langchain langchain_community pypdf pymupdf langchain_openai langchain_huggingface

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installati

In [2]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

#/content/drive/MyDrive/edurekaai/_data/rag/*.txt

Mounted at /content/drive


## Document Structure in LangChain


In [ ]:
from langchain_core.documents import Document
from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

In [ ]:
# Sample Document Structure
doc=Document(
    page_content="This is my document. It is the second document",
    metadata={
        "id": 2,
        "source": "The LangChain Papers",
        "page": 2,
        "author": "John Doe",
        "date": "2022-01-01",
        "custom_field": "foo"
    }
)

print(type(doc))
print(doc)
print(f"content: {doc.page_content}")
print(f"metadata: {doc.metadata}")

### Importantance of Metadata:
- Filtering docs effectively
- Tracking doc source
- Provides additional context

## Document Loaders
It takes care of loading documents from file system and scrapping web contents.

In [ ]:
document = Document(
            page_content="Hello, world!",
            metadata={"source": "https://example.com",
                      "author": "John Doe"}
        )

print(type(document))
print(f"Metadata: {document.metadata}")
print(f"Content: {document.page_content}")


In [ ]:
from langchain.document_loaders import TextLoader

#loader
loader = TextLoader('/content/drive/MyDrive/edurekaai/_data/rag/texts/python.txt')
docs = loader.load()

print(type(docs))
print(f"Loaded {len(docs)} documents")
print(f"Content preview: {docs[0].page_content[:100]}")
print(f"Content preview: {docs[0].metadata}")

In [ ]:
from langchain.document_loaders import DirectoryLoader, TextLoader

# Load files
loader = DirectoryLoader(
    '/content/drive/MyDrive/edurekaai/_data/rag/texts',
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True,
    loader_kwargs={'encoding': 'utf-8'}
)

# Actually load the documents
text_docs = loader.load()

print(type(text_docs))
print(f"Loaded {len(text_docs)} documents")

# Display document info
for i, doc in enumerate(text_docs):
    print(f"\nDocument {i+1}")
    print(f"Metadata: {doc.metadata}")
    print(f"Content preview: {doc.page_content[:200]}")  # show first 200 chars


In [ ]:
# PDF loader
from langchain.document_loaders import DirectoryLoader, PyMuPDFLoader
# PyMuPDFLoader is used when PDF has images. Otherwise, you can use PyPDFLoader
# TextLoader treats one file as one document.
# PDFLoader treats each page in the PDF as one document.

# Load all PDF files in the directory
loader = DirectoryLoader(
    '/content/drive/MyDrive/edurekaai/_data/rag/pdfs',
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_docs = loader.load()

print(type(pdf_docs))
print(f"Loaded {len(pdf_docs)} documents")

# Print document previews
for i, doc in enumerate(pdf_docs):
    print(f"\nDocument {i+1}")
    print(f"Content preview: {doc.page_content[:100]}")
    print(f"Metadata: {doc.metadata}")


## Document Splitter
Document Splitter splits the document into chunks. Chunk are list of tokens or words.

In [ ]:
from langchain.document_loaders import TextLoader

#loader
loader = TextLoader('/content/drive/MyDrive/edurekaai/_data/rag/texts/python.txt')
docs = loader.load()

In [ ]:
# Character Text Spliter

from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(separator=' ',
                                      chunk_size=100,
                                      chunk_overlap=20,
                                      length_function=len)

for i, doc in enumerate(docs):
    chunks = char_splitter.split_text(doc.page_content)
    print(f"No of chunks in document {i+1}: {len(chunks)}")
    print(f"\nFirst chunk: {chunks[0]}")
    print(f"------------------------------------")
    print(f"\nSecond chunk: {chunks[1]}")
    print(f"------------------------------------")
    print(f"\nLast chunk: {chunks[-1]}")

In [ ]:
# Recursive Character Text spliter
# It recursively tries multiple splitting strategies  ('\n', ' ', ', ', '.')

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter( separators=['\n\n', '\n', ' ', ',', '.'],
                                      chunk_size=100,
                                      chunk_overlap=20,
                                      length_function=len)

for i, doc in enumerate(docs):
    chunks = splitter.split_text(doc.page_content)
    print(f"No of chunks in document {i+1}: {len(chunks)}")
    print(f"\nFirst chunk: {chunks[0]}")
    print(f"------------------------------------")
    print(f"\nSecond chunk: {chunks[1]}")
    print(f"------------------------------------")
    print(f"\nLast chunk: {chunks[-1]}")

In [ ]:
# TokenTextSplitter
# It recursively tries multiple splitting strategies  ('\n', ' ', ', ', '.')

from langchain_text_splitters import TokenTextSplitter

splitter = TokenTextSplitter(
                             chunk_size=100,
                            chunk_overlap=20,
                            length_function=len)

for i, doc in enumerate(docs):
    chunks = splitter.split_text(doc.page_content)
    print(f"No of chunks in document {i+1}: {len(chunks)}")
    print(f"\nFirst chunk: {chunks[0]}")
    print(f"------------------------------------")
    print(f"\nSecond chunk: {chunks[1]}")
    print(f"------------------------------------")
    print(f"\nLast chunk: {chunks[-1]}")

## Embedding
Convert the chunks into vector embeddings.

In [ ]:
import os
from langchain.embeddings import OpenAIEmbeddings

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
##Note: these embedding models run the input through a neural network to generate one VECTOR for the entire input chunk.
## the chunk could be one word or multiple words.

# embed_query() generates one embedding for the entire text.
single_text = "Hello world"
single_embedding = embeddings.embed_query(single_text)
print(f"Emedding vector dimesion: {len(single_embedding)}")
print(f"Embedding vector: {single_embedding}")

# embed_documents() generates multiple embeddings.
multiple_text = """
The sky painted itself in shades of gold as the sun dipped below the horizon.
A gentle breeze carried the scent of rain and earth, whispering promises of renewal.
Children’s laughter echoed through the streets, fading into the rhythm of crickets.
In that quiet moment, time seemed to pause, holding its breath between day and night.
It was a reminder that even endings can be beautiful beginnings.
"""
multiple_embeddings = embeddings.embed_documents(multiple_text)
print(f"Number of embeddings: {len(multiple_embeddings)}")
print(f"Dimension of first embedding vector: {len(multiple_embeddings[0])}")
# print(f"Embedding vectors: {multiple_embeddings}")
# multiple_embeddings

## Embedding using HuggingFace Model

In [3]:
import os
from langchain_huggingface import HuggingFaceEmbeddings

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# embed_query() generates one embedding for the entire text.
single_text = "Hello world"
single_embedding = hf_embeddings.embed_query(single_text)
print(f"Emedding vector dimesion: {len(single_embedding)}")
print(f"Embedding vector: {single_embedding}")

# embed_documents() generates multiple embeddings.
# It is 2 dimesional array. (embedding_count x vector_size)
multiple_text = """
The sky painted itself in shades of gold as the sun dipped below the horizon.
A gentle breeze carried the scent of rain and earth, whispering promises of renewal.
Children’s laughter echoed through the streets, fading into the rhythm of crickets.
In that quiet moment, time seemed to pause, holding its breath between day and night.
It was a reminder that even endings can be beautiful beginnings.
"""
multiple_embeddings = hf_embeddings.embed_documents(multiple_text)
print(f"Number of embeddings: {len(multiple_embeddings)}")
print(f"Dimension of first embedding vector: {len(multiple_embeddings[0])}")
# for embedding in multiple_embeddings:
#   print(f"Embedding vector: {embedding}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Emedding vector dimesion: 384
Embedding vector: [-0.03447728976607323, 0.03102319873869419, 0.0067349751479923725, 0.026109017431735992, -0.03936200216412544, -0.16030250489711761, 0.06692400574684143, -0.006441437639296055, -0.047450508922338486, 0.014758866280317307, 0.07087535411119461, 0.05552755296230316, 0.019193312153220177, -0.02625133842229843, -0.010109523311257362, -0.026940515264868736, 0.022307416424155235, -0.022226650267839432, -0.14969263970851898, -0.017493100836873055, 0.007676243782043457, 0.05435231328010559, 0.003254458773881197, 0.031725961714982986, -0.08462141454219818, -0.029405998066067696, 0.05159568414092064, 0.048124074935913086, -0.0033148014917969704, -0.05827920511364937, 0.04196930304169655, 0.022210700437426567, 0.1281888335943222, -0.02233896777033806, -0.011656248942017555, 0.06292837858200073, -0.03287629783153534, -0.09122606366872787, -0.031175322830677032, 0.05269954353570938, 0.04703483730554581, -0.08420302718877792, -0.030056210234761238, -0.0

## Vector Store + dB

In [5]:
!pip install qdrant_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 27.1 MB/s eta 0:00:00


In [6]:
#Qdrant db
import os
from qdrant_client import QdrantClient
from google.colab import userdata

os.environ["QDRANT_KEY"] = userdata.get("QDRANT_KEY")
os.environ["QDRANT_URI"] = userdata.get("QDRANT_URI")

qdrant_client = QdrantClient(
    url=os.environ["QDRANT_URI"],
    api_key=os.environ["QDRANT_KEY"],
)

print(qdrant_client.get_collections())

collections=[]


In [7]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


In [9]:
import os
from google.colab import userdata
os.environ["PINECONE_KEY"] = userdata.get("PINECONE_KEY")

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_KEY"])

#Create an index
index_name = "developer-quickstart-py"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )


## Data Ingestion
It takes care of loading documents, splitting them into chunks, creating embedding, and then storing them in vector dB.